___

# <font color= #EF4444> **Speech Radicalization Analysis** </font>
#### <font color= #f0565e> `Deep Learning`</font>
<Strong> Sofía Maldonado, Oscar Josué Rocha & Viviana Toledo </Strong>

_12/05/2026._

___

# <font color= #EF4444> **Import Libraries & Data** </font>

In [10]:
# General
import numpy as np
import pandas as pd
from datasets import Dataset

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

# Modeling
import torch
from transformers import AutoModelForSequenceClassification

In [2]:
MODEL_NAME = "vinai/bertweet-base"

In [3]:
df = pd.read_csv('../data/processed/speech_classified.csv')
df = df[['clean_text', 'annotation_label']]
df

,clean_text,annotation_label
0,"china, this is what you are known for: animalr...",Hate Speech
1,catabuserschina visa workers and their employe...,Hate Speech
2,china madeinchina chinatravel china doesn't ha...,Extreme Speech
3,"they are ugly psychopaths, monsters!!!",Extreme Speech
4,and its culture of cruelty and brutality,Hate Speech
...,...,...
6213,its time for our weekly economic update on isr...,Hate Speech
6214,any doubt....,Hate Speech
6215,not this republican. neveragain allow another ...,Hate Speech
6216,"that's one way to filter your feed, i guess. p...",Enraged Speech


In [4]:
X = df['clean_text'].tolist()
y = df['annotation_label'].tolist()

# <font color= #EF4444> **Preprocessing** </font>

To start, let's map the labels to use as future reference:

In [5]:
# Initialize label encoder
label_encoder = LabelEncoder()

# Fit label encoder
y_label = label_encoder.fit_transform(y)

# Num of labels
num_labels = len(label_encoder.classes_)

# Show label mapping
mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
mapping

{np.str_('Dangerous Speech'): np.int64(0),
 np.str_('Enraged Speech'): np.int64(1),
 np.str_('Extreme Speech'): np.int64(2),
 np.str_('Hate Grammar'): np.int64(3),
 np.str_('Hate Speech'): np.int64(4),
 np.str_('Horror Grammar'): np.int64(5)}

Before tokenizing, let's do a train-test-split and convert the data to hugging face datasets:

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y_label, test_size=0.2, random_state=69, stratify=y)

In [7]:
train_dataset = Dataset.from_dict({
    "text": X_train,
    "label": y_train
})

test_dataset = Dataset.from_dict({
    "text": X_test,
    "label": y_test
})

Next, it's needed to tokenize the data:

In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

def tokenize(batch):
    return tokenizer(
        batch['text'],
        padding = 'max_length',
        truncation = True,
        max_length = 128,
    )

train_dataset = train_dataset.map(tokenize, batched = True)
test_dataset = test_dataset.map(tokenize, batched = True)

[transformers] emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0
Map: 100%|██████████| 1244/1244 [00:00<00:00, 4669.40 examples/s]


In [9]:
# Preprare data for pytorch loading
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# <font color= #EF4444> **Modeling** </font>

In [11]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels = num_labels)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 34830.24it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing 